In [1]:
# Cell 1 — Install dependencies (run once)
import subprocess
subprocess.run(["pip3", "install", "nltk", "scikit-learn", "pandas"], check=True)

import pandas as pd
import numpy as np
import ast
import re
from collections import Counter

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')   # ADD — required for NLTK 3.8+
nltk.download('vader_lexicon')
nltk.download('omw-1.4')     # ADD — required by WordNetLemmatizer on some systems

print("All imports done.")


[notice] A new release of pip available: 22.3 -> 26.2
[notice] To update, run: pip3 install --upgrade pip


All imports done.


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/huntstar/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/huntstar/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /Users/huntstar/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/huntstar/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/huntstar/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/huntstar/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Cell 2 — Load CSV and verify expected columns exist
CSV_PATH = "/Users/huntstar/Projects/Zomato_project/Restaurant-Analytics-Predictive-Intelligence-System/RAG/outputs/zomato_rag_cleaned.csv"   # <-- change path if needed

df = pd.read_csv(CSV_PATH)

required = ['name', 'location', 'reviews_list', 'reviews_available',
            'dish_liked', 'dish_liked_available', 'is_rate_imputed', 'rate']
assert all(c in df.columns for c in required), "Missing columns! Check your CSV."

print(f"Total rows loaded: {len(df)}")
print(df['reviews_available'].value_counts())
print(df['dish_liked_available'].value_counts())

Total rows loaded: 51717
reviews_available
True     44122
False     7595
Name: count, dtype: int64
dish_liked_available
False    28078
True     23639
Name: count, dtype: int64


In [3]:
# Cell 3 — Split dataframe
df_has_reviews = df[df['reviews_available'] == True].copy()
df_no_reviews  = df[df['reviews_available'] == False].copy()

print(f"Rows with reviews   : {len(df_has_reviews)}")   # expect ~44,122
print(f"Rows without reviews: {len(df_no_reviews)}")    # expect ~7,595

Rows with reviews   : 44122
Rows without reviews: 7595


In [4]:
# Cell 4 — Parse the stringified list-of-tuples in reviews_list
def parse_reviews(raw):
    try:
        parsed = ast.literal_eval(raw)
        texts = []
        for item in parsed:
            if isinstance(item, (tuple, list)) and len(item) >= 2:
                text = str(item[1])
                text = re.sub(r'^RATED\s*\n?', '', text, flags=re.IGNORECASE).strip()
                if text:
                    texts.append(text)
        return texts
    except:
        return []

df_has_reviews['review_list_parsed'] = df_has_reviews['reviews_list'].apply(parse_reviews)
df_has_reviews['total_reviews']      = df_has_reviews['review_list_parsed'].apply(len)

print(df_has_reviews['total_reviews'].describe())

count    44122.000000
mean        29.864467
std         76.983573
min          0.000000
25%          2.000000
50%          5.000000
75%         14.000000
max       1765.000000
Name: total_reviews, dtype: float64


In [5]:
# Cell 5 — Explode so each review is its own row (needed for VADER per-review scoring)
df_exploded = df_has_reviews[['name', 'location', 'review_list_parsed']].copy()
df_exploded = df_exploded.explode('review_list_parsed').rename(
    columns={'review_list_parsed': 'raw_review_text'}
)
df_exploded = df_exploded[
    df_exploded['raw_review_text'].notna() &
    df_exploded['raw_review_text'].ne('')
].reset_index(drop=True)

print(f"Total individual reviews to process: {len(df_exploded)}")

Total individual reviews to process: 1317680


In [6]:
# Cell 6 — Text cleaning for keyword extraction (TF-IDF)
# Note: VADER in Cell 7 runs on raw_review_text, NOT this cleaned version
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

def clean_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df_exploded['clean_review_text'] = df_exploded['raw_review_text'].apply(clean_text)
print("Text cleaning done.")
print(df_exploded[['raw_review_text', 'clean_review_text']].head(3))

Text cleaning done.
                                     raw_review_text  \
0  A beautiful place to dine in.The interiors tak...   
1  I was here for dinner with my family on a week...   
2  Its a restaurant near to Banashankari BDA. Me ...   

                                   clean_review_text  
0  beautiful place dine interior take back mughal...  
1  dinner family weekday restaurant completely em...  
2  restaurant near banashankari bda along office ...  


In [7]:
# Cell 7 — VADER sentiment scoring per review
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    scores   = sia.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        label = 'Positive'
    elif compound <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'
    return compound, label

df_exploded[['sentiment_compound', 'sentiment_label']] = df_exploded['raw_review_text'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

print(df_exploded['sentiment_label'].value_counts())

sentiment_label
Positive    1054080
Negative     189818
Neutral       73782
Name: count, dtype: int64


In [8]:
# Cell 8 — Aggregate per-review scores up to restaurant level

# INSTEAD OF joining all reviews, take a stratified sample:
def sample_reviews(review_series, max_words=350):
    """
    Take a balanced sample of reviews that fits within embedding token budget.
    Picks from positive, negative, and neutral reviews proportionally.
    """
    reviews = list(review_series)
    if not reviews:
        return ''
    # Shuffle to avoid always picking the first N
    import random
    random.seed(42)
    random.shuffle(reviews)
    
    result = []
    word_count = 0
    for r in reviews:
        words = len(r.split())
        if word_count + words > max_words:
            break
        result.append(r)
        word_count += words
    return ' '.join(result)
  # clean_review          = ('clean_review_text', lambda x: ' '.join(x)),
agg = df_exploded.groupby(['name', 'location']).agg(
    avg_sentiment_score   = ('sentiment_compound', 'mean'),
    positive_review_count = ('sentiment_label', lambda x: (x == 'Positive').sum()),
    negative_review_count = ('sentiment_label', lambda x: (x == 'Negative').sum()),
    neutral_review_count  = ('sentiment_label', lambda x: (x == 'Neutral').sum()),
    clean_review = ('clean_review_text', sample_reviews),
    total_reviews_parsed  = ('sentiment_compound', 'count'),
    avg_review_length     = ('raw_review_text', lambda x: x.apply(lambda t: len(t.split())).mean()),
    avg_character_count   = ('raw_review_text', lambda x: x.apply(len).mean()),
).reset_index()

# # Dominant sentiment = whichever count is highest
# def dominant_label(row):
#     return max(
#         [('Positive', row.positive_review_count),
#          ('Negative', row.negative_review_count),
#          ('Neutral',  row.neutral_review_count)],
#         key=lambda x: x[1]
#     )[0]

# agg['dominant_sentiment']  = agg.apply(dominant_label, axis=1)
# Tier based on avg_sentiment_score — much more useful than majority vote
def sentiment_tier(avg_score):
    if avg_score >= 0.5:
        return 'Highly Positive'
    elif avg_score >= 0.2:
        return 'Positive'
    elif avg_score >= -0.05:
        return 'Mixed'
    elif avg_score >= -0.2:
        return 'Negative'
    else:
        return 'Highly Negative'

agg['dominant_sentiment'] = agg['avg_sentiment_score'].apply(sentiment_tier)

# Add positive_ratio — a 0-to-1 number, much easier for RAG to filter on
agg['positive_ratio'] = (
    agg['positive_review_count'] / agg['total_reviews_parsed']
).round(3)

agg['review_quality_flag'] = agg['total_reviews_parsed'].apply(
    lambda n: 'low' if n < 3 else 'normal'
)

print(f"Restaurants aggregated: {len(agg)}")
print(agg[['name', 'dominant_sentiment', 'avg_sentiment_score', 'total_reviews_parsed']].head(5))

Restaurants aggregated: 9840
            name dominant_sentiment  avg_sentiment_score  total_reviews_parsed
0   #FeelTheROLL    Highly Positive             0.823950                     8
1     #L-81 Cafe    Highly Positive             0.941903                    35
2  #Vibes Restro    Highly Positive             0.740400                     3
3        #refuel           Positive             0.345900                     6
4    1 Fahreheit    Highly Positive             0.780000                     2


In [9]:
# Cell 9 — Extract top keywords per restaurant from review text using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.85,
    sublinear_tf=True,
)
tfidf_matrix  = vectorizer.fit_transform(agg['clean_review'])
feature_names = vectorizer.get_feature_names_out()

print(f"Vocabulary size: {len(feature_names)}")  # just vocab size here, column doesn't exist yet

def top_keywords(row_idx, n=7):
    row    = tfidf_matrix[row_idx]
    scores = zip(feature_names, row.toarray()[0])
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return [w for w, s in sorted_scores[:n] if s > 0]

agg['review_keywords_list'] = [top_keywords(i) for i in range(tfidf_matrix.shape[0])]

# NOW it's safe to print — column exists
print("TF-IDF done. Sample:")
print(agg[['name', 'review_keywords_list']].head(5))

Vocabulary size: 8000
TF-IDF done. Sample:
            name                               review_keywords_list
0   #FeelTheROLL  [egg chicken, chicken roll, roll, roll really,...
1     #L-81 Cafe  [item tasty, expected food, corn sandwich, lit...
2  #Vibes Restro  [kashmiri, forget try, price pay, quantity qua...
3        #refuel  [ordered schezwan, paneer sandwich, increase q...
4    1 Fahreheit  [waffle, icecream, cream roll, good awesome, i...


In [10]:
# Cell 10 — Extract dish keywords from dish_liked column (only where dish_liked_available = True)
dish_df = df[df['dish_liked_available'] == True][['name', 'location', 'dish_liked']].copy()

def parse_dishes(raw):
    if not raw or str(raw).strip() in ('', 'nan', '[]'):
        return []
    return [d.strip().lower() for d in str(raw).split(',') if d.strip()]

dish_df['dish_list'] = dish_df['dish_liked'].apply(parse_dishes)

# Global dish frequency to rank dish keywords
all_dishes       = [d for dishes in dish_df['dish_list'] for d in dishes]
global_dish_freq = Counter(all_dishes)

def top_dish_keywords(dish_list, n=5):
    ranked = sorted(set(dish_list), key=lambda d: global_dish_freq.get(d, 0), reverse=True)
    return ranked[:n]

dish_df['dish_keywords_list'] = dish_df['dish_list'].apply(top_dish_keywords)

dish_agg = dish_df[['name', 'location', 'dish_keywords_list']].drop_duplicates(['name', 'location'])

# Merge dish keywords into agg
agg = agg.merge(dish_agg, on=['name', 'location'], how='left')
agg['dish_keywords_list'] = agg['dish_keywords_list'].apply(
    lambda x: x if isinstance(x, list) else []
)

print(f"Restaurants with dish keywords: {(agg['dish_keywords_list'].apply(len) > 0).sum()}")
print(agg[['name', 'dish_keywords_list']].head(3))

Restaurants with dish keywords: 4565
            name dish_keywords_list
0   #FeelTheROLL                 []
1     #L-81 Cafe          [burgers]
2  #Vibes Restro                 []


In [11]:
# Cell 11 — Merge both keyword sources (dish keywords take priority)
def merge_keywords(review_kws, dish_kws):
    seen   = set()
    merged = []
    for kw in (dish_kws + review_kws):   # dish first = higher retrieval priority
        if kw not in seen:
            seen.add(kw)
            merged.append(kw)
    return ', '.join(merged)

agg['keywords']               = agg.apply(lambda r: merge_keywords(r.review_keywords_list, r.dish_keywords_list), axis=1)
agg['dish_keywords']          = agg['dish_keywords_list'].apply(lambda x: ', '.join(x) if x else None)
agg['review_keywords']        = agg['review_keywords_list'].apply(lambda x: ', '.join(x) if x else None)
agg['keywords_dish_enriched'] = agg['dish_keywords_list'].apply(lambda x: len(x) > 0)

print("Keywords merged. Sample:")
print(agg[['name', 'keywords', 'keywords_dish_enriched']].head(5))

Keywords merged. Sample:
            name                                           keywords  \
0   #FeelTheROLL  egg chicken, chicken roll, roll, roll really, ...   
1     #L-81 Cafe  burgers, item tasty, expected food, corn sandw...   
2  #Vibes Restro  kashmiri, forget try, price pay, quantity qual...   
3        #refuel  pasta, mocktails, sandwiches, thick shakes, or...   
4    1 Fahreheit  waffle, icecream, cream roll, good awesome, ic...   

   keywords_dish_enriched  
0                   False  
1                    True  
2                   False  
3                    True  
4                   False  


In [12]:
# Cell 12 — Merge NLP results back onto original full dataframe and export to CSV
nlp_cols = [
    'name', 'location',
    'avg_sentiment_score', 'dominant_sentiment',
    'positive_ratio', 
    'positive_review_count', 'negative_review_count', 'neutral_review_count',
    'total_reviews_parsed', 'clean_review',
    'avg_review_length', 'avg_character_count',
    'review_quality_flag',
    'keywords', 'review_keywords', 'dish_keywords', 'keywords_dish_enriched'
]

# enriched = df.merge(agg[nlp_cols], on=['name', 'location'], how='left')

# OUTPUT_PATH = "restaurant_reviews_enriched_imputed.csv"
# enriched.to_csv(OUTPUT_PATH, index=False)

# print(f"Total rows exported        : {len(enriched)}")
# print(f"NLP processed (has reviews): {enriched['avg_sentiment_score'].notna().sum()}")
# print(f"NLP null (no reviews)      : {enriched['avg_sentiment_score'].isna().sum()}")
# print(f"Saved to: {OUTPUT_PATH}")

# Mark source BEFORE merge so we can track inherited vs direct NLP
agg['nlp_source'] = 'direct'  # these rows had reviews_list themselves

# enriched = df.merge(agg[nlp_cols + ['nlp_source']], on=['name', 'location'], how='left')


# # Rows that got NLP via name+location match but had no reviews of their own
# # reviews_available=False but avg_sentiment_score is not null = inherited
# enriched['nlp_source'] = enriched.apply(
#     lambda r: r['nlp_source'] if pd.notna(r['nlp_source'])
#               else ('inherited' if pd.notna(r.get('avg_sentiment_score')) else 'none'),
#     axis=1
# )

enriched = df.merge(agg[nlp_cols + ['nlp_source']], on=['name', 'location'], how='left')

# Correct the 511 rows that got NLP via name+location match
# but had no reviews_list of their own — these are 'inherited', not 'direct'
mask = (enriched['reviews_available'] == False) & (enriched['nlp_source'] == 'direct')
enriched.loc[mask, 'nlp_source'] = 'inherited'

# Handle remaining unmatched rows (the 7084 with no reviews and no twin)
enriched['nlp_source'] = enriched['nlp_source'].fillna('none')

OUTPUT_PATH = "restaurant_reviews_enriched_imputed.csv"
enriched.to_csv(OUTPUT_PATH, index=False)

# Updated print block — now shows all three categories clearly
print(f"Total rows exported : {len(enriched)}")
print(f"\nnlp_source breakdown:")
print(enriched['nlp_source'].value_counts())
print(f"\nCross-check:")
print(f"  reviews_available=True  rows : {(enriched['reviews_available']==True).sum()}")
print(f"  nlp_source=direct       rows : {(enriched['nlp_source']=='direct').sum()}")
print(f"  nlp_source=inherited    rows : {(enriched['nlp_source']=='inherited').sum()}")
print(f"  nlp_source=none         rows : {(enriched['nlp_source']=='none').sum()}")
print(f"\nSaved to: {OUTPUT_PATH}")

Total rows exported : 51717

nlp_source breakdown:
nlp_source
direct       44122
none          7084
inherited      511
Name: count, dtype: int64

Cross-check:
  reviews_available=True  rows : 44122
  nlp_source=direct       rows : 44122
  nlp_source=inherited    rows : 511
  nlp_source=none         rows : 7084

Saved to: restaurant_reviews_enriched_imputed.csv


In [13]:
# Cell 13 — Sanity check on output
out = pd.read_csv("restaurant_reviews_enriched_imputed.csv")

print("Shape:", out.shape)
print("\nColumn list:")
print(list(out.columns))

print("\nSentiment distribution:")
print(out['dominant_sentiment'].value_counts())

print("\nDish enriched count:")
print(out['keywords_dish_enriched'].value_counts())

print("\nSample row:")
print(out[out['reviews_available'] == True][['name', 'dominant_sentiment', 'keywords']].head(3))

Shape: (51717, 35)

Column list:
['name', 'location', 'rest_type', 'cuisines', 'dish_liked', 'reviews_list', 'approx_cost(for two people)', 'rate', 'votes', 'online_order', 'book_table', 'menu_item', 'listed_in(type)', 'is_rate_imputed', 'cuisines_available', 'dish_liked_available', 'reviews_available', 'menu_available', 'rag_document', 'avg_sentiment_score', 'dominant_sentiment', 'positive_ratio', 'positive_review_count', 'negative_review_count', 'neutral_review_count', 'total_reviews_parsed', 'clean_review', 'avg_review_length', 'avg_character_count', 'review_quality_flag', 'keywords', 'review_keywords', 'dish_keywords', 'keywords_dish_enriched', 'nlp_source']

Sentiment distribution:
dominant_sentiment
Highly Positive    30770
Positive            7328
Mixed               3731
Highly Negative     1781
Negative            1023
Name: count, dtype: int64

Dish enriched count:
keywords_dish_enriched
True     24023
False    20610
Name: count, dtype: int64

Sample row:
              name d

In [14]:
# Cell 14: Explode keywords for Tableau Word Cloud
import pandas as pd

kw_rows = []
for _, row in enriched[enriched['keywords'].notna()].iterrows():
    for kw in str(row['keywords']).split(','):
        kw = kw.strip()
        if kw:
            kw_rows.append({
                'name': row['name'],
                'location': row['location'],
                'dominant_sentiment': row['dominant_sentiment'],
                'avg_sentiment_score': row['avg_sentiment_score'],
                'keyword': kw
            })

kw_df = pd.DataFrame(kw_rows)
kw_df.to_csv('keywords_exploded.csv', index=False)
print(f"Total keyword rows: {len(kw_df)}")

Total keyword rows: 407887
